In [0]:
# ============================================================
# LOAD GLOBAL PROJECT CONFIGURATION
# ============================================================

# Loads reusable constants, API parameters,
# naming conventions and audit configuration
# shared across the project.

# MAGIC %run ./utils_config

In [0]:
%run ./utils_config

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 99 Utils — Câmara API Client
# MAGIC
# MAGIC **Notebook:** `utils_api_client`
# MAGIC
# MAGIC Provides reusable functions to request data from the Câmara dos Deputados Open Data API.
# MAGIC
# MAGIC This notebook centralizes API access logic used by Bronze ingestion notebooks.
# MAGIC
# MAGIC ## Responsibilities
# MAGIC
# MAGIC - Execute HTTP GET requests against Câmara Open Data API
# MAGIC - Apply timeout and retry strategy
# MAGIC - Standardize API response handling
# MAGIC - Support resilient Bronze ingestion
# MAGIC - Preserve error details for auditability
# MAGIC
# MAGIC ## Naming Standard
# MAGIC
# MAGIC - Table and field names use Portuguese mnemonics.
# MAGIC - Comments and documentation are written in English.
# MAGIC - Utility functions use descriptive English names for reuse and readability.

# COMMAND ----------

import time
import requests
from typing import Optional, Dict, Any

# COMMAND ----------

# MAGIC %run ./utils_config

# COMMAND ----------

def fetch_camara_api_data(
    endpoint_path: str,
    query_params: Optional[Dict[str, Any]] = None,
    request_timeout_seconds: int = API_REQUEST_TIMEOUT_SECONDS,
    max_retry_attempts: int = API_MAX_RETRY_ATTEMPTS,
    retry_sleep_seconds: int = API_RETRY_SLEEP_SECONDS,
) -> Dict[str, Any]:
    """
    Requests data from a Câmara dos Deputados Open Data API endpoint.

    Parameters
    ----------
    endpoint_path : str
        API endpoint path, for example: "/deputados".
    query_params : dict, optional
        Query parameters sent to the API request.
    request_timeout_seconds : int
        Maximum request timeout in seconds.
    max_retry_attempts : int
        Maximum number of retry attempts.
    retry_sleep_seconds : int
        Waiting time between retry attempts.

    Returns
    -------
    dict
        JSON response returned by the API.

    Raises
    ------
    Exception
        Raises the last captured exception when all retry attempts fail.
    """

    request_url = f"{CAMARA_API_BASE_URL}{endpoint_path}"
    last_error = None

    for attempt_number in range(1, max_retry_attempts + 1):
        try:
            api_response = requests.get(
                request_url,
                params=query_params,
                timeout=request_timeout_seconds,
            )

            api_response.raise_for_status()

            return api_response.json()

        except Exception as error:
            last_error = error

            print(
                f"[WARNING] API request failed "
                f"| endpoint={endpoint_path} "
                f"| attempt={attempt_number}/{max_retry_attempts} "
                f"| error={str(error)}"
            )

            if attempt_number < max_retry_attempts:
                time.sleep(retry_sleep_seconds * attempt_number)

    raise last_error

# COMMAND ----------

def extract_response_records(
    api_response: Dict[str, Any],
    data_field_name: str = API_RESPONSE_DATA_FIELD,
) -> list:
    """
    Extracts the data records list from a Câmara API JSON response.

    Parameters
    ----------
    api_response : dict
        Full API JSON response.
    data_field_name : str
        Field name that contains the data records.

    Returns
    -------
    list
        List of records found in the API response.
    """

    if api_response is None:
        return []

    return api_response.get(data_field_name, [])

# COMMAND ----------

def check_endpoint_availability(
    endpoint_path: str,
    query_params: Optional[Dict[str, Any]] = None,
) -> bool:
    """
    Checks whether an API endpoint is available.

    Parameters
    ----------
    endpoint_path : str
        API endpoint path.
    query_params : dict, optional
        Query parameters sent to the API request.

    Returns
    -------
    bool
        True when the endpoint returns a valid response, otherwise False.
    """

    try:
        fetch_camara_api_data(
            endpoint_path=endpoint_path,
            query_params=query_params,
            max_retry_attempts=1,
        )
        return True

    except Exception as error:
        print(
            f"[WARNING] Endpoint availability check failed "
            f"| endpoint={endpoint_path} "
            f"| error={str(error)}"
        )
        return False

# COMMAND ----------

print("utils_api_client loaded successfully.")

utils_api_client loaded successfully.


Project configuration loaded successfully.
Project: brazil_legislative_analytics
Catalog: brazil_legislative_analytics
Environment: dev
Default legislatures: [56, 57]
Analysis years: [2022, 2023, 2024, 2025, 2026]
